<a href="https://colab.research.google.com/github/DeepLabCut/DeepLabCut/blob/master/examples/COLAB/COLAB_maDLC_TrainNetwork_VideoAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

# DeepLabCut 用于您的多动物项目！

一些有用的链接：

- [DeepLabCut 的 GitHub: github.com/DeepLabCut/DeepLabCut](https://github.com/DeepLabCut/DeepLabCut)
- [DeepLabCut 的文档：多动物项目用户指南](https://deeplabcut.github.io/DeepLabCut/docs/maDLC_UserGuide.html)


![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1628180434489-T0RIWEJJU0FJVOT6FNVD/maDLC.png?format=800w)

本教程（Notebook）演示了如何在多动物项目中使用基于云端的 GPU 来完成以下操作：
- 创建一个多动物训练集
- 训练一个网络模型
- 评估网络模型
- 分析新的视频（Novel videos）
- 组合动物和轨迹片段（tracklets）
- 创建质量检查图表！

### 本教程假设您已经有了一个包含已标记数据的 DLC 项目文件夹，并且已将其上传到您自己的 Google 云端硬盘（Google Drive）中。

本教程展示了使用 DeepLabCut 处理您自己项目的必要步骤。

这里展示了最简单的代码实现方式，但许多函数都包含额外的功能，因此请务必查阅 GitHub 上的官方文档。我们还推荐您查阅我们的预印本论文，该论文涵盖了 maDLC 的科学原理。

**Lauer 等人 2021：** https://www.biorxiv.org/content/10.1101/2021.04.30.442096v1

以下是翻译结果：

## 首先，前往 "Runtime" -> "change runtime type" -> 选择 "Python3"，然后选择 "GPU"

由于 COLAB 环境已更新到 CUDA 12.X 和 Python 3.11，我们需要以一种特殊的方式安装 DeepLabCut 和 TensorFlow，以确保 TensorFlow 能够正确连接到 GPU。

In [ ]:
# this will take a couple of minutes to install all the dependencies!
!pip install --pre deeplabcut

（请务必在继续之前，**点击上方显示的“RESTART RUNTIME”（重启运行时）按钮！**）您会在上方单元格的输出中看到此按钮 ^。

In [2]:
import deeplabcut

## 链接您的 Google Drive（包含已标记数据）：

- 此代码假设您已在本地安装了 DeepLabCut，创建了一个项目，并提取和标记了帧。请务必“检查标签”以确认您对数据感到满意。因为这些帧是训练网络的唯一依据。💪 您可以在此处找到执行此操作的所有文档：[deeplabcut.github.io/DeepLabCut](https://deeplabcut.github.io/DeepLabCut/README.html)
- 接下来，将您的 DLC 项目文件夹放入您的 Google Drive 中——即，将名为 "Project-YourName-TheDate" 的文件夹复制到 Google Drive 中。
- 然后，单击下方单元格中的“运行”按钮，将此 Notebook 连接到您的 Google Drive：

In [ ]:
# Now, let's link to your GoogleDrive. Run this cell and follow the authorization instructions:
# (We recommend putting a copy of the github repo in your google drive if you are using the demo "examples")

from google.colab import drive

drive.mount("/content/drive")

## 接下来，编辑以下几个项目，然后点击运行（Run）：

您**必须在 `config.yaml` 文件中**将项目路径编辑为您 Google Drive 链接所指向的路径！通常情况下，这个路径将是：`/content/drive/My Drive/yourProjectFolderName`

In [ ]:
# PLEASE EDIT THIS:
project_folder_name = "MontBlanc-Daniel-2019-12-16"
video_type = "mp4" #, mp4, MOV, or avi, whatever you uploaded!

# No need to edit this, we are going to assume you put videos you want to analyze
# in the "videos" folder, but if this is NOT true, edit below:
videofile_path = [f"/content/drive/My Drive/{project_folder_name}/videos/"]
print(videofile_path)

# The prediction files and labeled videos will be saved in this `labeled-videos` folder
# in your project folder; if you want them elsewhere, you can edit this;
# if you want the output files in the same folder as the videos, set this to an empty string.
destfolder = f"/content/drive/My Drive/{project_folder_name}/labeled-videos"

#No need to edit this, as you set it when you passed the ProjectFolderName (above):
path_config_file = f"/content/drive/My Drive/{project_folder_name}/config.yaml"
print(path_config_file)

# This creates a path variable that links to your Google Drive project

## 创建多动物训练数据集：

- 更多信息可以在[文档中找到](https://deeplabcut.github.io/DeepLabCut/docs/maDLC_UserGuide.html#create-training-dataset)
- 请检查以下文本，如有需要请进行编辑，然后点击运行（此过程可能需要一些时间）：

In [ ]:
# OPTIONAL LEARNING: did you know you can check what each function does by running with a ?
deeplabcut.create_multianimaltraining_dataset?

In [ ]:
# ATTENTION:
# Which shuffle do you want to create and train?
shuffle = 1 # Edit if needed; 1 is the default.

deeplabcut.create_multianimaltraining_dataset(
    path_config_file,
    Shuffles=[shuffle],
    net_type="dlcrnet_ms5",
    engine=deeplabcut.Engine.PYTORCH,
)

## 开始训练：

此函数针对训练数据集的特定**洗牌顺序**（或称随机打乱顺序）来训练网络。更多信息可以在 [官方文档](https://deeplabcut.github.io/DeepLabCut/docs/maDLC_UserGuide.html#train-the-network) 中找到。

In [ ]:
# Let's also change the display and save_epochs just in case Colab takes away
# the GPU... If that happens, you can reload from a saved point using the
# `snapshot_path` argument to `deeplabcut.train_network`:
#   deeplabcut.train_network(..., snapshot_path="/content/.../snapshot-050.pt")

# Typically, you want to train to ~200 epochs. We set the batch size to 8 to
# utilize the GPU's capabilities.

# More info and there are more things you can set:
#   https://deeplabcut.github.io/DeepLabCut/docs/standardDeepLabCut_UserGuide.html#g-train-the-network

deeplabcut.train_network(
    path_config_file,
    shuffle=shuffle,
    save_epochs=5,
    epochs=200,
    batch_size=8,
)

# This will run until you stop it (CTRL+C), or hit "STOP" icon, or when it hits the end.

请注意，当您按下 "STOP" 时，您会收到一个 `KeyboardInterrupt` “错误”！别担心！:)

## 开始评估：

- 首先，我们评估姿态估计的性能。
- 此函数会针对特定 `shuffle`/`shuffles` 在特定状态下（或在数据集的**所有状态**上）评估训练好的模型，并将结果存储在 `evaluation-results-pytorch` 目录下子文件夹中的 `.5` 和 `.csv` 文件中。
- 如果得分图（scoremaps）看起来不准确，请不要继续进行轨迹组装（tracklet assembly）；请考虑 (1) 添加更多数据，(2) 添加更多身体部位（bodyparts）！
- 更多信息可以在[文档中找到](https://deeplabcut.github.io/DeepLabCut/docs/maDLC_UserGuide.html#evaluate-the-trained-network)

以下是继续操作之前希望看到的示例：

![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1590535809087-X655WY9W1MW1MY1I7DHE/ke17ZwdGBToddI8pDm48kBoswZhKnUtAF7-bTXgw67EUqsxRUqqbr1mOJYKfIPR7LoDQ9mXPOjoJoqy81S2I8N_N4V1vUb5AoIIIbLZhVYxCRW4BPu10St3TBAUQYVKc5tTP1cnANTUwNNPnYFjIp6XbP9N1GxIgAkxvBVqt0UvLpPHYwvNQTwHg8f_Zu8ZF/evaluation.png?format=1000w)

In [ ]:
# Let's evaluate first:
deeplabcut.evaluate_network(path_config_file, Shuffles=[shuffle], plotting=True)

# plot a few scoremaps:
deeplabcut.extract_save_all_maps(path_config_file, shuffle=shuffle, Indices=[0, 1, 2, 3])

如果这些图像、数字和地图看起来效果不佳，请不要继续。你应该增加你标记的帧（frames）的多样性和数量，然后重新创建一个训练数据集并重新训练模型！

## 开始分析视频：
此函数用于分析新的视频。用户可以从评估结果中选择最佳的模型，并在 `config.yaml` 文件中为变量 **snapshotindex** 指定正确的快照索引。否则，系统将默认使用最新的快照来分析视频。

分析结果将以 pickle 文件的形式存储在视频所在的同一目录下。

In [ ]:
print("Start Analyzing my video(s)!")
#EDIT OPTION: which video(s) do you want to analyze? You can pass a path or a folder:
# currently, if you run "as is" it assumes you have a video in the DLC project video folder!

deeplabcut.analyze_videos(
    path_config_file,
    videofile_path,
    shuffle=shuffle,
    videotype=video_type,
    auto_track=False,
    destfolder=destfolder,
)

可选的：现在您有权在跟踪动物之前检查原始检测结果。要执行此操作，请传递一个视频路径：

In [ ]:
##### PROTIP: #####
## look at the output video; if the pose estimation (i.e. key points)
## don't look good, don't proceed with tracking - add more data to your training set and re-train!

# EDIT: let's check a specific video (PLEASE EDIT VIDEO PATH):
specific_videofile = "/content/drive/MyDrive/DeepLabCut_maDLC_DemoData/MontBlanc-Daniel-2019-12-16/videos/short.mov"

# Don't edit:
deeplabcut.create_video_with_all_detections(
    path_config_file, [specific_videofile], shuffle=shuffle, destfolder=destfolder,
)

如果最终生成的视频（以 `full.mp4` 结尾）效果不佳，我们强烈建议您添加更多数据并重新训练模型。相关信息请参阅[官方文档中的此处](https://deeplabcut.github.io/DeepLabCut/docs/maDLC_UserGuide.html#decision-break-point)。

## 接下来，我们将使用数据驱动的最优图方法来组装动物：

在视频分析过程中，动物是使用最优图（optimal graph）来组装的，这与我们论文中的“数据驱动”方法相匹配（图示改编自 Lauer 等人 2021 年的研究）。

![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1626266017809-XO6NX84QB4FBAZGOTCEY/fig3.jpg?format=400w)

最优图是在执行 `evaluate_network` 函数时计算出来的——所以请确保不要跳过这一步！

**注意**：你可以设置你预期看到的动物数量，因此请检查、编辑，然后点击运行：

In [ ]:
#Check and edit:
num_animals = 4 # How many animals do you expect to find?
track_type= "box" # box, skeleton, ellipse
#-- ellipse is recommended, unless you have a single-point MA project, then use BOX!

# Optional:
# imagine you tracked a point that is not useful for assembly,
# like a tail tip that is far from the body, consider dropping it for this step (it's still used later)!
# To drop it, uncomment the next line TWO lines and add your parts(s):

# bodypart= 'Tail_end'
# deeplabcut.convert_detections2tracklets(path_config_file, videofile_path, videotype=VideoType, shuffle=shuffle, overwrite=True, ignore_bodyparts=[bodypart])

# OR don't drop, just click RUN:
deeplabcut.convert_detections2tracklets(
    path_config_file,
    videofile_path,
    videotype=video_type,
    shuffle=shuffle,
    track_method=track_type,
    destfolder=destfolder,
    overwrite=True,
)

deeplabcut.stitch_tracklets(
    path_config_file,
    videofile_path,
    shuffle=shuffle,
    track_method=track_type,
    n_tracks=num_animals,
    destfolder=destfolder,
)

现在我们来过滤数据，以移除任何微小的抖动（或称“噪声”）：

In [ ]:
deeplabcut.filterpredictions(
    path_config_file,
    videofile_path,
    shuffle=shuffle,
    videotype=video_type,
    track_method=track_type,
    destfolder=destfolder,
)

## 创建您的轨迹图：

In [ ]:
deeplabcut.plot_trajectories(
    path_config_file,
    videofile_path,
    videotype=video_type,
    shuffle=shuffle,
    track_method=track_type,
    destfolder=destfolder,
)

现在您可以查看 `plot-poses` 文件，并检查 `plot-likelihood.png` 图片。您可能需要在 `config.yaml` 配置文件中更改 `"p-cutoff"` 的值，以确保视频中只绘制置信度高的点。例如，可以设置为 **0.8** 或 **0.9** 左右。当前的默认值是 **0.4**。

## 创建带标签的视频：
此函数用于可视化目的，可用于创建网络预测标签的 .mp4 格式视频。此视频将保存在原始视频所在的同一目录下。

In [ ]:
deeplabcut.create_labeled_video(
    path_config_file,
    videofile_path,
    shuffle=shuffle,
    color_by="individual",
    videotype=video_type,
    save_frames=False,
    filtered=True,
    track_method=track_type,
    destfolder=destfolder,
)